# Unix-terminal. Работа с текстами

## Мотивация

Данные приходят текстом: логи обучения, CSV с разметкой, конфиги, вывод чужих скриптов. Пока файл маленький, его можно открыть в редакторе; но лог обучения на 2 ГБ редактор не откроет, а на удалённом сервере через SSH нет ни IDE, ни файлового менеджера — только терминал.

Три вопроса возникают там постоянно: **где лежит нужный файл**, **есть ли в нём нужный текст** и **как достать из него конкретные значения**. На них отвечают `find`, `grep` и регулярные выражения — этому и посвящено занятие. Всё остальное — сортировка, подсчёты, сравнение файлов — надстройки над той же идеей потока текста.

Отдельная причина — сила композиции. Каждая утилита делает одно действие, а конвейер из четырёх коротких команд заменяет скрипт на полстраницы, который ещё надо написать и отладить.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Почему утилиты такие мелкие — это не случайность, а решение, принятое в
1970-х. Дуг Макилрой сформулировал его так: «пишите программы, которые делают
одну вещь и делают её хорошо; пишите программы, которые работают вместе;
пишите программы, обрабатывающие текстовые потоки, потому что это
универсальный интерфейс». Конвейер `|` он же и продавил в Unix в 1973 году.

Отсюда же ответ на вопрос «почему не написать одну большую программу
`analyze`»: набор из десяти маленьких утилит комбинируется в тысячи сценариев,
а большая программа умеет ровно то, что в неё заложил автор.

</details>

> **Как устроен семинар.** Ноутбук состоит из двух частей: **демонстрация**
> (разделы 1–5) — её показывают на занятии и повторяют за преподавателем, и
> **справочник** (разделы 6–9) — его не демонстрируют, он нужен при решении
> задач.
>
> Свёрнутые блоки **🎙 Заметка преподавателя** — это то, что рассказывается
> вслух на занятии. Разворачивайте их потом, при подготовке к защите.
>
> Все команды выполняются **на вашей виртуальной машине, в домашнем каталоге**.
> Скопируйте туда данные семинара и запускайте Jupyter из `~/seminar-03/` —
> иначе относительные пути не найдут файлы. Каталог `/tmp` не используем: он
> вычищается при перезагрузке.

In [ ]:
%%bash
ls assets                                       # какие файлы нам дали
wc -l assets/train.log assets/contacts.txt      # и сколько в них строк

## 1. Редакторы: `nano`, `vim`, `ed`

Начинаем с редакторов, потому что это первое, обо что спотыкаются на сервере: `git commit` без `-m`, правка конфига, беглый просмотр чужого скрипта — всё упирается в то, какой редактор там оказался.

**`nano`** — простой: печатаете текст как есть, подсказки внизу экрана, `^` означает `Ctrl`.

- `Ctrl+O`, затем `Enter` — сохранить;
- `Ctrl+X` — выйти;
- `Ctrl+W` — поиск.

**`vim`** есть почти везде и работает в режимах. После запуска вы в *нормальном* режиме, где буквы — это команды, а не текст.

- `i` — перейти в режим вставки, `Esc` — вернуться в нормальный;
- `:w` — сохранить, `:q` — выйти, `:wq` — сохранить и выйти, `:q!` — выйти без сохранения;
- `/текст` — поиск, `n` — следующее совпадение;
- `dd` — удалить строку, `u` — отменить действие.

**`ed`** — строчный редактор без экрана: команды применяются к строкам, файл не отображается. Сегодня им правят файлы редко, но он остаётся последним доступным редактором в аварийном окружении, где терминал не умеет управлять экраном.

`vimdiff old new` открывает два файла рядом с подсветкой различий; переход между окнами — `Ctrl+W`, затем стрелка.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`ed` выглядит издевательством ровно до того момента, пока не поймёшь, что он
писался для телетайпа: экрана не было, каждая строка вывода печаталась на
бумаге. Отсюда и лаконичность — на ошибочную команду `ed` отвечает одним
символом `?`, потому что бумага стоила денег.

Из `ed` выросло почти всё, чем мы пользуемся: команда `g/re/p` («globally
search for regular expression and print») стала отдельной утилитой `grep`,
а редактор `vi` — это визуальный режим поверх той же идеи, потом ставший
`vim`.

Про «как выйти из vim»: это один из самых просматриваемых вопросов на Stack
Overflow за всю его историю. Смеяться будем ровно до первого раза, когда
`git commit` без `-m` откроет vim на сервере.

</details>

#### ❓ **Вопрос**: Вы открыли файл в `vim`, начали печатать — и текст не появился, зато курсор прыгает по экрану. Что произошло и как теперь выйти без сохранения?

<details>

<summary><strong>Ответ</strong></summary>

`vim` стартует в нормальном режиме: набранные буквы выполняются как команды. Для ввода текста нужен `i`. Чтобы выйти без сохранения — `Esc`, затем `:q!` и `Enter`.

</details>

## 2. Просмотр: `head`, `tail`, `wc`

Открывать редактором большой файл незачем — сначала на него смотрят по частям.

- `head -n N file` — первые `N` строк;
- `tail -n N file` — последние `N` строк;
- `wc file` — счётчики: `-l` строки, `-w` слова, `-c` байты, `-m` символы.

`tail -n +N` — не то же самое, что `head`: он печатает файл начиная со строки `N` и до конца. Так отбрасывают заголовок CSV.

In [ ]:
%%bash
head -n 3 assets/train.log            # начало файла: с чего запуск начался
echo '--- последние 2 строки ---'
tail -n 2 assets/train.log            # конец файла: чем всё закончилось

In [ ]:
%%bash
# Байты и символы различаются: кириллица в UTF-8 занимает по два байта на букву.
wc -l -w -c assets/train.log
wc -m assets/contacts.txt

`less file` открывает файл для просмотра, не загружая его целиком: `Space` — страница вперёд, `b` — назад, `/текст` — поиск, `n` — следующее совпадение, `G` — в конец, `q` — выход.

`tail -f file` не завершается, а печатает новые строки по мере их появления — так смотрят за работающим обучением. Прерывается по `Ctrl+C`.

#### ❓ **Вопрос**: Файл содержит 19 строк. Что напечатают `head -n 5 assets/train.log` и `tail -n +5 assets/train.log`?

<details>

<summary><strong>Ответ</strong></summary>

`head -n 5` напечатает строки 1–5. `tail -n +5` напечатает строки с 5-й по 19-ю, то есть 15 строк. Форма `-n +N` задаёт номер строки, с которой начинается вывод, а не количество строк.

</details>

## 3. `find`: найти файл

Первый вопрос на чужой машине — где вообще лежит то, что нужно. Отвечает `find`: он обходит дерево каталогов и отбирает файлы **по свойствам** — имени, типу, размеру, времени изменения.

```bash
find ПУТЬ УСЛОВИЯ
```

- `-name '*.py'` — по имени (кавычки обязательны, иначе шаблон раскроет оболочка);
- `-type f` — только файлы, `-type d` — только каталоги;
- `-size +200c` — больше 200 байт, `-size -1M` — меньше мегабайта;
- `-mtime -1` — изменён меньше суток назад;
- `-delete`, `-exec КОМАНДА {} \;` — что сделать с найденным.

In [ ]:
%%bash
find assets/project -type f -name '*.py'       # кавычки обязательны: иначе * раскроет оболочка

In [ ]:
%%bash
find assets/project -type f -size +200c | sort # +200c — «строго больше 200 байт»

Передавать список файлов дальше по конвейеру нужно аккуратно: имена могут содержать пробелы. Пара `-print0` и `xargs -0` разделяет пути нулевым байтом, который в имени файла встретиться не может.

Почему -print0 и xargs -0

In [ ]:
%%bash
find assets/project -type f -name '*.csv' -print0 | xargs -0 wc -l   # имя с пробелом уцелело
echo '--- то же самое без -print0 ---'
find assets/project -type f -name '*.csv' | xargs wc -l || true      # путь распался по пробелу

#### ❓ **Вопрос**: В дереве есть файл `old data.csv`. Что произойдёт с `find ... -name '*.csv' | xargs wc -l` и почему помогает `-print0`?

<details>

<summary><strong>Ответ</strong></summary>

`xargs` по умолчанию разделяет аргументы пробелами, поэтому путь распадётся на `assets/project/data/old` и `data.csv` — обоих файлов не существует, и `wc` напечатает ошибки, а итог окажется занижен. `-print0` разделяет пути нулевым байтом, `xargs -0` читает их так же, и имя с пробелом остаётся целым.

</details>

## 4. `grep`: найти текст внутри файлов

`find` отвечает, какие файлы подходят по имени и размеру. `grep` отвечает на другой вопрос — **в каких файлах есть нужный текст**. Эти два вопроса постоянно путают:

grep ищет внутри файлов, find — по их свойствам

Базовая форма — `grep ШАБЛОН файл`. Дальше разберём флаги по одному: каждый отвечает на свой вопрос.

In [ ]:
%%bash
# -n добавляет номер строки: по нему потом открывают файл в редакторе.
grep -n ERROR assets/train.log

In [ ]:
%%bash
# -c печатает только счётчик — сами строки не выводятся.
grep -c ERROR assets/train.log

In [ ]:
%%bash
# -v инвертирует условие: остаётся то, где совпадения НЕТ.
grep -v INFO assets/train.log

In [ ]:
%%bash
grep -i -c error assets/train.log     # -i: ERROR, Error и error — одно и то же
grep -w -c INFO assets/train.log      # -w: только отдельное слово, INFORMATION не подойдёт

`-r` (или `-R`) идёт по дереву каталогов и печатает совпадения с именем файла и номером строки.

In [ ]:
%%bash
# Порядок обхода каталога не гарантирован, поэтому результат сортируем.
grep -rn TODO assets/project | sort

#### ❓ **Вопрос**: Чем `grep -c ERROR file` отличается от `grep ERROR file | wc -l`, и когда они дадут разные числа?

<details>

<summary><strong>Ответ</strong></summary>

На этих данных — ничем: оба считают строки с совпадением. Разойдутся они, если добавить `-o`: тогда `grep` печатает каждое совпадение отдельной строкой, и `wc -l` посчитает совпадения, а не строки. В строке с двумя `ERROR` `-c` даст 1, а `-o | wc -l` — 2.

</details>

## 5. Регулярные выражения

Шаблон `grep` — это не подстрока, а **регулярное выражение**: описание того, как выглядит искомый текст. Это главный навык занятия: он одинаково работает в `grep`, в редакторе, в Python и в поиске вашей IDE.

Кирпичики:

| Запись | Что означает |
|---|---|
| `.` | любой одиночный символ |
| `[0-9]` | одна цифра; `[а-яё]` — одна строчная буква |
| `[^,]` | любой символ, **кроме** запятой |
| `^` | начало строки |
| `$` | конец строки |
| `\|` | «или» — альтернатива |

`grep` по умолчанию понимает *базовый* синтаксис (BRE), где `+`, `?`, `|` — обычные символы. Расширенный синтаксис включается флагом `-E`. Практическое правило: пишете что-то сложнее подстроки — берите `-E`.

In [ ]:
%%bash
grep 'loss=[0-9]+' assets/train.log | wc -l     # без -E плюс — обычный символ: 0 совпадений
grep -E 'loss=[0-9]+' assets/train.log | wc -l  # с -E плюс стал оператором: 10 строк

### Якоря: начало и конец строки

`^` и `$` не совпадают ни с одним символом — они привязывают шаблон к краю строки. Без них `grep` найдёт совпадение где угодно внутри.

In [ ]:
%%bash
grep -cE '^Иванов' assets/contacts.txt        # строки, НАЧИНАЮЩИЕСЯ с фамилии
grep -cE 'example\.org$' assets/contacts.txt  # строки, ЗАКАНЧИВАЮЩИЕСЯ этим адресом

Точка внутри `example\.org` экранирована обратным слэшем: без него `.` означала бы «любой символ», и шаблон совпал бы и с `exampleXorg`.

### Квантификаторы: сколько раз повторяется

- `?` — ноль или один раз;
- `+` — один и больше;
- `*` — ноль и больше;
- `{n}` — ровно `n` раз, `{n,m}` — от `n` до `m`.

In [ ]:
%%bash
printf 'файл\nфайл1\nфайл123\n' | grep -E 'файл[0-9]?$'   # ? — цифры нет или одна
echo '--- + требует хотя бы одну ---'
printf 'файл\nфайл1\nфайл123\n' | grep -E 'файл[0-9]+$'   # + — одна и больше

### Альтернатива и слова целиком

`|` внутри `-E` означает «или», а скобки группируют выражение.

In [ ]:
%%bash
grep -cE 'Иванов|Орлов' assets/contacts.txt        # любая из двух фамилий
grep -w -c 'тел' assets/contacts.txt               # ровно слово «тел», а не «телефон»

### Разбор боевой задачи: найти номера паспортов

В выгрузке номер записан двумя способами — `4509 123456` и `45 09 654321`. Собираем шаблон из кирпичиков: две цифры, необязательный пробел, ещё две цифры, пробел, шесть цифр.

```
[0-9]{2} ?[0-9]{2} [0-9]{6}
```

Именно здесь `?` и `{n}` окупаются: одно выражение покрывает обе формы записи.

In [ ]:
%%bash
# -o печатает только совпавшую часть, а не всю строку с фамилией и телефоном.
grep -oE '[0-9]{2} ?[0-9]{2} [0-9]{6}' assets/contacts.txt

In [ ]:
%%bash
# -v с тем же шаблоном отвечает на обратный вопрос: у кого паспорта нет.
grep -vE '[0-9]{2} ?[0-9]{2} [0-9]{6}' assets/contacts.txt

#### ❓ **Вопрос**: Шаблон `[0-9]{4} [0-9]{6}` находит только `4509 123456`, а запись `45 09 654321` пропускает. Как починить и почему нельзя просто написать `.*`?

<details>

<summary><strong>Ответ</strong></summary>

Нужно разрешить необязательный пробел внутри серии: `[0-9]{2} ?[0-9]{2} [0-9]{6}`. Знак `?` относится к предыдущему элементу — к пробелу — и делает его необязательным. `.*` не подходит, потому что совпадёт с чем угодно между цифрами: в номер попадут фамилии, скобки и куски телефона, и вы получите мусор вместо данных.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Полезно проговорить, что регулярные выражения — это не «синтаксис grep», а
математический объект: конечный автомат, описанный Стивеном Клини в 1951 году.
Отсюда и звёздочка — «звезда Клини».

Практический вывод из этого: у регулярных выражений есть предел. Вложенные
структуры — скобки, HTML, JSON — они разбирать не умеют в принципе, потому что
автомат не помнит, сколько скобок уже открыто. Канонический ответ на Stack
Overflow «нельзя распарсить HTML регуляркой» — не вредность, а теорема.
Поэтому JSON мы будем читать библиотекой, а не `grep`.

</details>

---

# Справочная часть

Дальше — то, что не показывают на занятии: пригодится при решении задач и при
подготовке к защите.

## 6. `sort` и `uniq`: посчитать и упорядочить

`sort` сортирует строки:

- `-n` — как числа, а не как текст (иначе `10` окажется раньше `9`);
- `-r` — по убыванию;
- `-u` — убрать повторы;
- `-t СИМВОЛ` — разделитель полей, `-k N,N` — по какому полю сортировать.

`uniq` схлопывает **соседние** одинаковые строки: `-c` добавляет счётчик, `-d` оставляет только повторяющиеся. Поэтому `uniq` почти всегда идёт после `sort`.

Конвейер `sort | uniq -c | sort -k1,1nr` — стандартный способ построить частотную таблицу:

Конвейер: sort, uniq -c, sort

In [ ]:
%%bash
# -t, — поля разделены запятой; -k3,3nr — по 3-му полю как число по убыванию.
tail -n +2 assets/students.csv | sort -t, -k3,3nr | head -n 3

In [ ]:
%%bash
# tail -n +2 отбрасывает заголовок, cut берёт колонку с группой,
# sort ставит одинаковые рядом, uniq -c считает, последний sort — по убыванию числа.
tail -n +2 assets/students.csv | cut -d, -f2 | sort | uniq -c | sort -k1,1nr -k2,2

In [ ]:
%%bash
printf 'b\na\nb\n' | uniq -c            # два b не рядом — посчитались раздельно
echo '--- то же самое, но после sort ---'
printf 'b\na\nb\n' | sort | uniq -c     # sort поставил их рядом — теперь 2 b

## 7. Колонки: `cut`, `tr`, `column`

`cut -d СИМВОЛ -f N` вырезает поля: `-d` задаёт разделитель, `-f` — номера полей (`-f1,3`, `-f2-4`).

`tr SET1 SET2` заменяет символы по одному: `tr ',' '\t'` меняет запятые на табуляции, `tr '[:upper:]' '[:lower:]'` приводит текст к нижнему регистру, `-d` удаляет символы, `-s` сжимает повторы.

`column -t -s,` выравнивает поля в таблицу — удобно, чтобы просто посмотреть CSV глазами.

Ограничение: `cut` не понимает кавычки. Для CSV, где запятая может стоять внутри поля `"Ivanov, Ivan"`, нужен настоящий CSV-парсер, а не `cut`.

In [ ]:
%%bash
cut -d, -f1,3 assets/students.csv | head -n 4          # только имя и балл
echo '--- та же таблица глазами ---'
column -t -s, assets/students.csv | head -n 4          # выравнивание по колонкам

## 8. `diff`: сравнение файлов

`diff old new` показывает, чем файлы различаются. Флаг `-u` даёт унифицированный формат — тот же, что в патчах и в `git diff`: строки со знаком `-` есть только в старом файле, со знаком `+` — только в новом.

Код возврата `diff` — часть ответа: `0` — файлы совпадают, `1` — различаются, `2` — произошла ошибка. Поэтому `diff` удобно ставить в проверки внутри скриптов.

Родственные инструменты: `diff -r dir1 dir2` сравнивает каталоги, `comm` сравнивает два *отсортированных* файла, `vimdiff` показывает различия в редакторе.

In [ ]:
%%bash
diff -u assets/config-old.txt assets/config-new.txt
echo "код возврата: $?"                # 1 — файлы различаются, это не ошибка

In [ ]:
%%bash
# -f берёт шаблоны из файла, -F сравнивает как текст, -x строку целиком,
# -v оставляет несовпавшие: получаем строки, которых в старом конфиге не было.
grep -Fxvf assets/config-old.txt assets/config-new.txt

#### ❓ **Вопрос**: Скрипт проверяет, что сгенерированный файл совпал с эталоном, через `diff -q got.txt expected.txt`. Как узнать результат проверки, если вывод не нужен?

<details>

<summary><strong>Ответ</strong></summary>

По коду возврата: `0` — файлы совпали, `1` — различаются, `2` — ошибка (например, файла нет). В скрипте это пишут как `if diff -q got.txt expected.txt > /dev/null; then ...`.

</details>

## 9. Аналоги на Rust

Классические утилиты переписывают заново — обычно на Rust. Причины практические: параллельный обход дерева, быстрый поиск, разумные значения по умолчанию и понятные сообщения об ошибках.

| Классика | Аналог | Что меняется |
|---|---|---|
| `grep -r` | `ripgrep` (`rg`) | быстрый рекурсивный поиск, по умолчанию пропускает пути из `.gitignore` и скрытые файлы |
| `find -name` | `fd` | короткий синтаксис: `fd '\.py$'` вместо `find . -name '*.py'` |
| `cat` | `bat` | подсветка синтаксиса и номера строк |
| `diff` | `delta` | читаемый цветной diff, в том числе для `git` |
| `coreutils` | `uutils coreutils` | те же `ls`, `wc`, `sort` с совместимыми флагами |
| — | `xsv`, `qsv` | настоящий CSV: колонки по именам, кавычки, статистика |

Главное расхождение в поведении: `rg` по умолчанию ищет не везде. Он пропускает скрытые файлы и, внутри git-репозитория, пути из `.gitignore` — это удобно в проекте и неожиданно, когда ищешь именно в артефактах сборки. Возвращают их флагами `--hidden` и `-uu`.

Совместимость флагов у аналогов неполная: в чужом скрипте или CI лучше оставлять классические утилиты, которые точно есть на любой машине.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Почему `ripgrep` действительно быстрее, а не «просто новее»: он обходит дерево
в несколько потоков, использует конечный автомат вместо возвратов и по
умолчанию работает в байтовом режиме, не пересчитывая каждую строку по
правилам локали. GNU grep, наоборот, обязан быть совместимым со всеми
странностями POSIX, а это стоит времени.

Здесь же полезно проговорить, что «быстро» и «правильно» — разные цели. В
чужом CI-скрипте `rg` может незаметно пропустить файлы из `.gitignore`, и
проверка окажется зелёной на пустом множестве. Поэтому в автоматике берут то,
что предсказуемо, а `rg` — для живого поиска руками.

</details>

#### ❓ **Вопрос**: `grep -rn TODO project/` нашёл 4 совпадения, а `rg -n TODO project/` — только 3. Кто прав?

<details>

<summary><strong>Ответ</strong></summary>

Оба работают верно. `rg` по умолчанию пропускает скрытые файлы и пути из `.gitignore`, поэтому одно совпадение он не показал. `grep -r` таких правил не знает и читает всё подряд. Чтобы вернуть пропущенное, используют `rg --hidden` и `rg -uu`.

</details>

## Дополнительно

### Порядок сортировки зависит от локали

`sort` сравнивает строки по правилам текущей локали: в UTF-8 регистр обычно игнорируется. `LC_ALL=C` переключает на сравнение по байтам — тогда все заглавные буквы идут раньше строчных.

Это важно, когда результат сравнивают с эталоном: одна и та же команда на машине разработчика и в CI может дать разный порядок строк. В скриптах, где порядок должен быть воспроизводимым, явно ставят `LC_ALL=C sort`.

In [ ]:
%%bash
printf 'apple\nApple\nbanana\nBanana\n' | sort             # по локали: регистр не важен
echo '--- LC_ALL=C ---'
printf 'apple\nApple\nbanana\nBanana\n' | LC_ALL=C sort    # по байтам: сначала заглавные

### Кодировки и переводы строк

`file file.txt` показывает, что программа думает о содержимом: кодировку и тип переводов строк. Файл из Windows содержит `\r\n`, и лишний `\r` попадает в конец каждого поля — из-за этого сравнение строк неожиданно перестаёт работать. Убирают его через `tr -d '\r'`. Перекодировать текст между кодировками умеет `iconv -f cp1251 -t utf-8`.

### Чего здесь нет

`sed` и `awk` — потоковый редактор и язык обработки таблиц — закрывают то, что неудобно делать через `grep`/`cut`: замены по шаблону и арифметику по колонкам. Это отдельная большая тема; в этом семинаре они не нужны, все задачи решаются разобранными утилитами.